# Liu2024 SignalJEPA PreLocal: Clean Marker-Aligned Evaluation

Independent per-trial preprocessing, exact marker-2 alignment, explicit inner validation, and exactly-once trial-level OOF artifacts. This notebook is a clean replacement experiment, not a rewrite of the historical 57% pipeline.

# 1. Setup

In [ ]:
import builtins
import hashlib
import json
import os
import platform
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch

WORKING_DIR = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
import liu2024_prelocal_clean as clean

mne.set_log_level('WARNING')
print(f'Python: {sys.version.split()[0]} | platform: {platform.platform()}')
print(f'Working directory: {WORKING_DIR}')

# 2. Configuration## 2.1 Liu2024 Clean Defaults

In [ ]:
print(f'Liu29 channels: {len(clean.LIU_EEG_NAMES)}')
print('Task window: exact marker-2 onset through 4.0 s post-onset')
print('Preprocessing scope: one independent 8 s trial at a time')

## 2.2 CONFIG

In [ ]:
CONFIG = {
    # Paths / run identity
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-sjepa-prelocal-clean'),
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'experiment_name': 'sjepa_prelocal_clean_structural_smoke',
    'config_note': 'Safe default: sub-01, two epochs; marker-aligned implementation smoke only.',

    # Dataset and preprocessing
    'subjects_to_use': [1],
    'exclude_subjects': [],
    'target_sfreq': 128,
    'mi_window_seconds': 4.0,
    'average_reference': True,
    'bandpass_hz': [0.5, 40.0],
    'normalization_mode': 'none',
    'normalization_eps': 1e-6,

    # Model
    'model_name': 'SignalJEPA_PreLocal',
    'pretrained_repo_id': 'braindecode/signal-jepa_without-chans',
    'pretrained_revision': '213876ea30f0764fd25c055efcb55d1d1652a371',
    'pretrained_checkpoint_path': None,
    'pretrained_checkpoint_sha256': None,
    'strategy': 'new',

    # Evaluation
    'cv_folds': 5,
    'cv_seed': 2026,
    'val_fraction': 0.2,
    'val_seed': 2026,

    # Training
    'batch_size': 16,
    'n_epochs': 2,
    'early_stopping_patience': 50,
    'early_stopping_threshold': 0.0,
    'learning_rate': 0.0005,
    'weight_decay': 0.0,

    # Reproducibility and diagnostics
    'seed': 2026,
    'set_seed': True,
    'collapse_threshold': 0.875,
}

In [ ]:
if CONFIG['strategy'] != 'new':
    raise ValueError('The clean notebook currently supports only adapter/head strategy=new.')
if CONFIG['mi_window_seconds'] != 4.0 or CONFIG['target_sfreq'] != 128:
    raise ValueError('The primary clean endpoint is locked to marker-relative 0-4 s at 128 Hz.')
print(json.dumps(CONFIG, indent=2))

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f'{timestamp}_{config_hash}'

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG['artifact_dir']) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'
_LOG_FILE_HANDLE = open(LOG_PATH, 'a', buffering=1, encoding='utf-8', errors='replace')

def _safe_write_text(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, 'encoding', None) or 'utf-8'
        stream.write(text.encode(encoding, errors='replace').decode(encoding, errors='replace'))

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop('sep', ' '); end = kwargs.pop('end', '\n'); flush = kwargs.pop('flush', False); file = kwargs.pop('file', None)
    message = sep.join(str(arg) for arg in args)
    target = sys.stdout if file is None else file
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ''
    _safe_write_text(target, stamped + end); _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    if flush:
        target.flush(); _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / 'config.json'
config_path.write_text(json.dumps(CONFIG, indent=2) + '\n')
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')
print(f'Config:     {config_path}')

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device('mps')
    if torch.cuda.is_available(): return torch.device('cuda')
    return torch.device('cpu')
DEVICE = resolve_device()
BASE_SEED = int(CONFIG['seed'])
if CONFIG['set_seed']: clean.seed_everything(BASE_SEED)
print(f'Using device: {DEVICE} | seed: {BASE_SEED}')

# 3. Load and Prepare Data## 3.1 Explicit MAT Loading and Marker Validation

In [ ]:
source_root = Path(CONFIG['source_extract_dir'])
paths = sorted(source_root.glob('sub-*/sub-*_task-motor-imagery_eeg.mat'))
requested = None if CONFIG['subjects_to_use'] is None else {int(x) for x in CONFIG['subjects_to_use']}
excluded = {int(x) for x in CONFIG['exclude_subjects']}
paths = [p for p in paths if (requested is None or clean.subject_id_from_path(p) in requested) and clean.subject_id_from_path(p) not in excluded]
if not paths: raise FileNotFoundError(f'No selected MAT files under {source_root}')
SUBJECT_DATA = {}
inventory_rows, marker_rows = [], []
for path in paths:
    subject = clean.load_subject(path)
    sid = str(subject['subject_id'])
    x, records = clean.preprocess_subject(subject, CONFIG)
    SUBJECT_DATA[sid] = {'x': x, 'y': subject['labels'].copy()}
    inventory_rows.append({'subject_id': sid, 'source_path': subject['path'], 'source_sha256': subject['source_sha256'], 'raw_shape': list(subject['raw'].shape), 'preprocessed_shape': list(x.shape), 'class_counts': np.bincount(subject['labels'], minlength=2).tolist()})
    marker_rows.extend(records)
SUBJECTS = sorted(SUBJECT_DATA, key=int)
subject_inventory_path = ARTIFACT_DIR / 'subject_inventory.csv'
marker_inventory_path = ARTIFACT_DIR / 'trial_marker_inventory.csv'
pd.DataFrame(inventory_rows).to_csv(subject_inventory_path, index=False)
pd.DataFrame(marker_rows).to_csv(marker_inventory_path, index=False)
print(f'Loaded subjects: {SUBJECTS}')
print(f'Marker onset range: {min(r["marker2_source_sample"] for r in marker_rows)}..{max(r["marker2_source_sample"] for r in marker_rows)}')

## 3.2 Preprocessing ContractEach 8-second trial is independently average-referenced, filtered, and resampled before marker-relative extraction. No filter or resampler can cross a trial or fold boundary.

## 3.3 Dataset ClassesThe clean helper uses explicit `TensorDataset` instances only after outer and inner indices are fixed.

## 3.4 Located DataSource hashes, exact markers, and preprocessed shapes are saved before model fitting.

# 4. Model## 4.1 Pinned PreLocal Adapter and Head

In [ ]:
probe_model, MODEL_AUDIT = clean.build_model(CONFIG, 512)
print(json.dumps(MODEL_AUDIT, indent=2))
del probe_model

## 4.2 DiagnosticsEvery fold records trainable names, tested-state hash, exact checkpoint epoch, and collapse diagnostics.

# 5. Training## 5.1 Explicit Inner Validation and Checkpointing

## 5.2 Within-Subject CV Runner

In [ ]:
FOLD_RESULTS = []
for sid in SUBJECTS:
    data = SUBJECT_DATA[sid]
    for split in clean.make_outer_splits(data['y'], CONFIG):
        print(f'Subject {sid} fold {split["fold_id"]}/{CONFIG["cv_folds"]}')
        result = clean.run_fold(int(sid), data['x'], data['y'], split, CONFIG, DEVICE)
        FOLD_RESULTS.append(result)
        print(f"  BA={result['balanced_accuracy']:.3f} best_epoch={result['best_epoch']} pred={result['prediction_histogram']}")

## 5.3 Run Completion Assertions

In [ ]:
expected_folds = len(SUBJECTS) * int(CONFIG['cv_folds'])
assert len(FOLD_RESULTS) == expected_folds
assert all(not row['outer_test_used_for_fit'] and not row['outer_test_used_for_selection'] for row in FOLD_RESULTS)
print(f'Completed {len(FOLD_RESULTS)} folds with no outer-test fitting or selection.')

# 6. Results## 6.1 Exact-Once OOF Aggregation

In [ ]:
labels_by_subject = {sid: SUBJECT_DATA[sid]['y'] for sid in SUBJECTS}
SUBJECT_METRICS, GLOBAL_METRICS, TRIAL_PREDICTIONS = clean.aggregate(FOLD_RESULTS, labels_by_subject)
print(json.dumps(GLOBAL_METRICS, indent=2))

## 6.2 Performance Visualizations

In [ ]:
subject_performance_plot_path = ARTIFACT_DIR / 'clean_prelocal_subject_performance.png'
fig, ax = plt.subplots(figsize=(max(6, len(SUBJECTS) * 0.7), 4))
values = [100 * SUBJECT_METRICS[s]['balanced_accuracy'] for s in SUBJECTS]
ax.bar(SUBJECTS, values, color='#2a6f97'); ax.axhline(50, color='black', ls='--', lw=1)
ax.set(xlabel='Subject', ylabel='Exactly-once OOF balanced accuracy (%)', ylim=(0, 100))
fig.tight_layout(); fig.savefig(subject_performance_plot_path, dpi=160); plt.close(fig)
global_performance_plot_path = ARTIFACT_DIR / 'clean_prelocal_global_performance.png'
fig, ax = plt.subplots(figsize=(4, 4)); ax.bar(['PreLocal'], [100 * GLOBAL_METRICS['mean_subject_balanced_accuracy']], color='#2a6f97'); ax.axhline(50, color='black', ls='--'); ax.set(ylabel='Mean subject BA (%)', ylim=(0, 100)); fig.tight_layout(); fig.savefig(global_performance_plot_path, dpi=160); plt.close(fig)
confusion_performance_plot_path = ARTIFACT_DIR / 'clean_prelocal_confusion_matrix.png'
cm = np.zeros((2, 2), dtype=int)
for row in FOLD_RESULTS: cm += np.asarray(row['confusion_matrix'])
fig, ax = plt.subplots(figsize=(4, 4)); ax.imshow(cm, cmap='Blues');
for i in range(2):
    for j in range(2): ax.text(j, i, str(cm[i, j]), ha='center', va='center')
ax.set(xlabel='Predicted', ylabel='True', xticks=[0, 1], yticks=[0, 1]); fig.tight_layout(); fig.savefig(confusion_performance_plot_path, dpi=160); plt.close(fig)

## 6.3 Experiment Summary

In [ ]:
single_class = sum(row['collapse_diagnostics']['single_class_prediction'] for row in FOLD_RESULTS)
near_collapse = sum(row['collapse_diagnostics']['near_collapse_7_of_8'] for row in FOLD_RESULTS)
print(f"Mean subject BA: {100 * GLOBAL_METRICS['mean_subject_balanced_accuracy']:.2f}%")
print(f'Single-class folds: {single_class}/{len(FOLD_RESULTS)} | >=7/8 majority: {near_collapse}/{len(FOLD_RESULTS)}')
print(f"Exactly-once OOF: {GLOBAL_METRICS['exact_once_oof_all_subjects']} | predictions={GLOBAL_METRICS['n_original_trial_predictions']}")

## 6.4 Model-Specific AnalysisSpatial adaptation remains available in each fold's tested-state hash and trainable-parameter audit; this clean baseline does not add posthoc channel selection.

## 6.5 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / 'cv_results.json'
subject_metrics_path = ARTIFACT_DIR / 'subject_metrics.json'
global_metrics_path = ARTIFACT_DIR / 'global_metrics.json'
trial_predictions_path = ARTIFACT_DIR / 'trial_predictions.csv'
split_indices_path = ARTIFACT_DIR / 'split_indices.json'
cv_results_path.write_text(json.dumps(FOLD_RESULTS, indent=2) + '\n')
subject_metrics_path.write_text(json.dumps(SUBJECT_METRICS, indent=2) + '\n')
global_metrics_path.write_text(json.dumps(GLOBAL_METRICS, indent=2) + '\n')
pd.DataFrame(TRIAL_PREDICTIONS).sort_values(['subject_id', 'trial_index']).to_csv(trial_predictions_path, index=False)
split_payload = [{k: row[k] for k in ['subject_id', 'fold_id', 'train_indices', 'inner_train_indices', 'validation_indices', 'test_indices']} for row in FOLD_RESULTS]
split_indices_path.write_text(json.dumps(split_payload, indent=2) + '\n')
module_path = Path(clean.__file__).resolve()
run_metadata = {
    'run_id': RUN_ID, 'artifact_dir': str(ARTIFACT_DIR), 'experiment_name': CONFIG['experiment_name'], 'config_note': CONFIG['config_note'],
    'subjects': SUBJECTS, 'n_channels': len(clean.LIU_EEG_NAMES), 'channel_names': clean.LIU_EEG_NAMES,
    'model_name': CONFIG['model_name'], 'pretrained_repo_id': CONFIG['pretrained_repo_id'], 'pretrained_revision': CONFIG['pretrained_revision'], 'pretrained_checkpoint_path': CONFIG.get('pretrained_checkpoint_path'), 'pretrained_checkpoint_sha256': CONFIG.get('pretrained_checkpoint_sha256'), 'strategy': CONFIG['strategy'],
    'preprocessing_contract': 'independent complete-trial average reference/filter/resample, exact marker-2 crop [0,4)s',
    'implementation_module': str(module_path), 'implementation_sha256': clean.sha256_file(module_path),
    'seed': BASE_SEED, 'cv_seed': CONFIG['cv_seed'], 'val_seed': CONFIG['val_seed'], 'global_metrics': GLOBAL_METRICS,
    'leakage_assertions': {'independent_trial_preprocessing': True, 'exact_marker_required': True, 'normalizer_fit_inner_train_only': True, 'outer_test_used_for_fit': False, 'outer_test_used_for_selection': False, 'exact_once_oof': True},
    'performance_artifacts': {'fold_results': str(cv_results_path), 'subject_metrics': str(subject_metrics_path), 'global_metrics': str(global_metrics_path), 'trial_predictions': str(trial_predictions_path), 'split_indices': str(split_indices_path), 'marker_inventory': str(marker_inventory_path), 'subject_inventory': str(subject_inventory_path), 'subject_plot': str(subject_performance_plot_path), 'global_plot': str(global_performance_plot_path), 'confusion_plot': str(confusion_performance_plot_path)},
}
run_metadata_path = ARTIFACT_DIR / 'run_metadata.json'
run_metadata_path.write_text(json.dumps(run_metadata, indent=2) + '\n')
print(f'CV results saved to:      {cv_results_path}')
print(f'Subject metrics saved to: {subject_metrics_path}')
print(f'Global metrics saved to:  {global_metrics_path}')
print(f'Run metadata saved to:    {run_metadata_path}')
print(f'\nAll artifacts in: {ARTIFACT_DIR}')
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass